# 윤정 추출 모델 — KoELECTRA 파인튜닝 (경로 B)

**목표**: 가정통신문 문장을 받아 카테고리 5개로 분류

- 모델: `monologg/koelectra-base-v3-discriminator`
- 데이터: `notices_labeled_v2.jsonl` (147문장, is_todo=True 102개)
- 카테고리 5개: 일정 / 준비물 / 제출 / 건강·안전 / 기타  
  (비용은 데이터가 2개뿐 → 정규식으로 별도 처리, 모델 학습에서 제외)
- 학습 시간: T4 GPU 기준 약 15~25분

**실행 방법**
1. `런타임` → `런타임 유형 변경` → **T4 GPU** 선택
2. 셀을 위에서부터 순서대로 실행 (`Shift + Enter`)
3. 마지막 셀에서 `koelectra-extractor.zip` 다운로드 → `model/extraction/checkpoints/` 에 압축 풀기

## 1. 라이브러리 설치
코랩에 기본 설치된 transformers는 버전이 옛날이라 새로 깔아요.

In [1]:
!pip install -q transformers==4.44.2 datasets==2.21.0 accelerate==0.34.0 evaluate==0.4.3 scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 126.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


## 2. GPU 확인
T4 GPU 가 잡혔는지 확인. CPU 만 보이면 런타임 유형을 바꿔야 해요.

In [2]:
import torch
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

CUDA 사용 가능: True
GPU 이름: Tesla T4


## 3. 학습 데이터 업로드
왼쪽 폴더 아이콘 → 파일 업로드 → `notices_labeled_v2.jsonl` 올리기.

In [3]:
import json
from collections import Counter

with open("notices_labeled_v2.jsonl", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

# 비용은 데이터가 2개뿐이라 학습 제외 → 정규식으로 처리할 예정
# is_todo=True 이고 비용이 아닌 문장만 학습 데이터로 사용
train_rows = [
    r for r in rows
    if r["is_todo"] and r["category"] != "비용"
]

print(f"전체 문장: {len(rows)}")
print(f"학습 사용: {len(train_rows)} (is_todo=True, 비용 제외)")
print("\n카테고리 분포:")
for k, v in Counter(r["category"] for r in train_rows).most_common():
    print(f"  {k}: {v}")

전체 문장: 147
학습 사용: 100 (is_todo=True, 비용 제외)

카테고리 분포:
  건강·안전: 35
  준비물: 22
  제출: 18
  일정: 18
  기타: 7


## 4. 라벨 인코딩 + Train/Val 분할
5개 카테고리를 정수 ID로 변환하고 8:2 로 나눠요.

데이터가 100개 정도로 적기 때문에 **stratified split** 으로 카테고리 비율을 맞춰서 나눠야 해요. 그냥 랜덤으로 나누면 검증셋에 어떤 카테고리가 0개일 수도 있어요.

In [4]:
from sklearn.model_selection import train_test_split

LABEL_LIST = ["일정", "준비물", "제출", "건강·안전", "기타"]
label2id = {l: i for i, l in enumerate(LABEL_LIST)}
id2label = {i: l for i, l in enumerate(LABEL_LIST)}

texts  = [r["sentence"] for r in train_rows]
labels = [label2id[r["category"]] for r in train_rows]

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels,   # 카테고리 비율 유지
)

print(f"학습셋: {len(train_texts)}, 검증셋: {len(val_texts)}")

학습셋: 80, 검증셋: 20


## 5. 토크나이저 + 데이터셋 만들기
KoELECTRA 가 이해할 수 있는 토큰 ID 로 변환해요. 한국어 문장이라 평균 30~80 토큰이라 max_length 128 이면 충분.

In [5]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "monologg/koelectra-base-v3-discriminator"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

train_ds = Dataset.from_dict({"text": train_texts, "label": train_labels}).map(encode, batched=True)
val_ds   = Dataset.from_dict({"text": val_texts,   "label": val_labels  }).map(encode, batched=True)

train_ds = train_ds.remove_columns(["text"])
val_ds   = val_ds.remove_columns(["text"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/61.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

## 6. 모델 + 평가 지표 + 클래스 가중치 정의

분류 문제라서 정확도 + Macro F1 두 가지를 본다.  
Macro F1 은 카테고리별 F1 의 평균 — 데이터 적은 카테고리도 동등하게 평가됨.

**클래스 불균형 문제**: 기타(7개) vs 건강·안전(35개) — 단순 학습 시 기타 F1=0.00 발생.  
`compute_class_weight('balanced')` 로 희귀 클래스에 높은 가중치를 부여하고,  
`WeightedTrainer` 에서 가중치를 적용한 CrossEntropyLoss 를 사용한다.

In [6]:
from transformers import AutoModelForSequenceClassification, Trainer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
)

# 클래스 불균형 보정 — 기타(7개)·준비물(22개) 편차가 커서 필수
_w = compute_class_weight("balanced", classes=np.arange(len(LABEL_LIST)), y=train_labels)
_class_weights = torch.tensor(_w, dtype=torch.float)


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fn = torch.nn.CrossEntropyLoss(weight=_class_weights.to(logits.device))
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }


print("클래스 가중치:", {l: round(w, 3) for l, w in zip(LABEL_LIST, _w)})

pytorch_model.bin:   0%|          | 0.00/452M [00:00<?, ?B/s]

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 7. 학습 실행

에폭 15회, 배치 16. 데이터 100개 기준 T4 GPU 약 20~30분.  
이전 대비 변경 사항:
- `num_train_epochs` 10→15 (기타 클래스 충분히 학습)
- `learning_rate` 3e-5→2e-5 (더 안정적 수렴)
- `warmup_ratio=0.1` + `lr_scheduler_type='cosine'` 추가
- `WeightedTrainer` 사용 (클래스 불균형 보정)

`load_best_model_at_end=True` 로 검증 F1 가장 높은 체크포인트를 자동 보존.

In [7]:
from transformers import TrainingArguments, DataCollatorWithPadding

args = TrainingArguments(
    output_dir="./koelectra-output",
    save_safetensors=False,
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,1.528592,0.350000,0.103704
2,1.552500,1.450895,0.350000,0.103704
3,1.552500,1.396795,0.350000,0.121739
4,1.423000,1.339281,0.450000,0.213333
5,1.423000,1.297201,0.600000,0.419048
6,1.298500,1.259855,0.700000,0.553333
7,1.298500,1.222655,0.750000,0.598797
8,1.215900,1.199617,0.750000,0.598797
9,1.215900,1.185620,0.750000,0.598797
10,1.164500,1.179376,0.750000,0.598797


TrainOutput(global_step=50, training_loss=1.3308836555480956, metrics={'train_runtime': 122.5146, 'train_samples_per_second': 6.53, 'train_steps_per_second': 0.408, 'total_flos': 52623628492800.0, 'train_loss': 1.3308836555480956, 'epoch': 10.0})

## 8. 최종 평가
검증셋에서 카테고리별 F1 확인. 발표 자료에 그대로 쓸 수치예요.

In [8]:
from sklearn.metrics import classification_report

preds = trainer.predict(val_ds)
y_pred = np.argmax(preds.predictions, axis=-1)
y_true = preds.label_ids

print(classification_report(
    y_true, y_pred,
    target_names=LABEL_LIST,
    digits=4,
    zero_division=0,
))

              precision    recall  f1-score   support

          일정     1.0000    1.0000    1.0000         4
         준비물     1.0000    0.2500    0.4000         4
          제출     1.0000    0.7500    0.8571         4
       건강·안전     0.5833    1.0000    0.7368         7
          기타     0.0000    0.0000    0.0000         1

    accuracy                         0.7500        20
   macro avg     0.7167    0.6000    0.5988        20
weighted avg     0.8042    0.7500    0.7093        20



## 9. 모델 저장
추론할 때 필요한 파일들(가중치 + 토크나이저 + 라벨 매핑)을 한 폴더에 모음.

In [9]:
OUTPUT_DIR = "./koelectra-extractor"
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# 라벨 정보 따로 저장 — predict.py 에서 읽어 씀
with open(f"{OUTPUT_DIR}/labels.json", "w", encoding="utf-8") as f:
    json.dump({"labels": LABEL_LIST, "id2label": id2label, "label2id": label2id}, f, ensure_ascii=False, indent=2)

print("저장 완료:", OUTPUT_DIR)
!ls -lh {OUTPUT_DIR}

저장 완료: ./koelectra-extractor
total 432M
-rw-r--r-- 1 root root 1.2K Apr 26 14:21 config.json
-rw-r--r-- 1 root root  343 Apr 26 14:21 labels.json
-rw-r--r-- 1 root root 431M Apr 26 14:21 pytorch_model.bin
-rw-r--r-- 1 root root  125 Apr 26 14:21 special_tokens_map.json
-rw-r--r-- 1 root root 1.3K Apr 26 14:21 tokenizer_config.json
-rw-r--r-- 1 root root 797K Apr 26 14:21 tokenizer.json
-rw-r--r-- 1 root root 5.5K Apr 26 14:21 training_args.bin
-rw-r--r-- 1 root root 258K Apr 26 14:21 vocab.txt


## 11. predict.py 재정의 — 버그 수정 3종

> `model/extraction/predict.py` 가 삭제 상태(`git: D`)라 여기서 다시 생성.

| # | 버그 | 원인 | 수정 위치 |
|---|------|------|----------|
| 1 | `학부모님 안녕하세요.` 가 TODO로 잡힘 | `NON_TODO_PATTERNS` 에 `안녕하십니까`만 있고 `안녕하세요` 미포함 | **predict.py** |
| 2 | 첫 베트남어 문장 어색 | 헤더·인사말이 NLLB에 합쳐진 채로 전달됨 | **predict.py** `split_sentences()` |
| 3 | 검수 상세 `원→won` 오탐 | 글로사리 매칭이 `'원'` substring | **translation_tts/run_mvp_pipeline.py** |

아래 셀을 실행하면 수정된 `predict.py` 가 현재 폴더에 생성됩니다.

In [ ]:
import pathlib

PREDICT_PY = '"""\nmodel/extraction/predict.py\n============================\n가정통신문 -> TodoItem 리스트 추출 (경로 B 하이브리드)\n\n파이프라인\n  [1] split_sentences()   문장 분리 (헤더/제목 줄 조기 차단)  <- Bug 2 수정\n  [2] is_likely_todo()    인사말·서명 등 1차 제외              <- Bug 1 수정\n  [3] extract_due_date()  정규식: 날짜·마감\n  [4] classify_category() KoELECTRA: 5개 카테고리 + 비용 정규식\n  [5] calc_importance()   카테고리 + due_date + 키워드 -> 점수\n"""\n\nimport os, sys, re, json\nfrom typing import Optional\n\nimport torch\nfrom transformers import AutoTokenizer, AutoModelForSequenceClassification\n\n_PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), "../.."))\nsys.path.insert(0, os.path.join(_PROJECT_ROOT, "backend/app/models"))\n\ntry:\n    from schemas import TodoItem, Category  # type: ignore\n    SCHEMAS_AVAILABLE = True\nexcept ImportError:\n    SCHEMAS_AVAILABLE = False\n    TodoItem = None\n    Category = None\n\n# ── 1. 모델 로드 ─────────────────────────────────────────────────────────────\nHF_REPO_ID = "yunjeong116/koelectra-extractor"\nHF_SUBFOLDER = "koelectra-extractor"\nLOCAL_CHECKPOINT_DIR = os.path.join(\n    os.path.dirname(__file__), "checkpoints/koelectra-extractor"\n)\n\n_tokenizer = None\n_model = None\n_id2label = None\n_device = "cuda" if torch.cuda.is_available() else "cpu"\n\n\ndef _load_model():\n    global _tokenizer, _model, _id2label\n    if _model is not None:\n        return\n    if os.path.exists(os.path.join(LOCAL_CHECKPOINT_DIR, "pytorch_model.bin")):\n        load_kwargs = {"pretrained_model_name_or_path": LOCAL_CHECKPOINT_DIR}\n        labels_path = os.path.join(LOCAL_CHECKPOINT_DIR, "labels.json")\n    else:\n        load_kwargs = {\n            "pretrained_model_name_or_path": HF_REPO_ID,\n            "subfolder": HF_SUBFOLDER,\n        }\n        from huggingface_hub import hf_hub_download\n        labels_path = hf_hub_download(\n            repo_id=HF_REPO_ID, filename=f"{HF_SUBFOLDER}/labels.json"\n        )\n    _tokenizer = AutoTokenizer.from_pretrained(**load_kwargs)\n    _model = AutoModelForSequenceClassification.from_pretrained(**load_kwargs)\n    _model.to(_device)\n    _model.eval()\n    with open(labels_path, encoding="utf-8") as f:\n        meta = json.load(f)\n    _id2label = {int(k): v for k, v in meta["id2label"].items()}\n\n\n# ── 2. 문장 분리 ─────────────────────────────────────────────────────────────\n# Bug 2 수정: 제목성 줄(헤더)을 split 단계에서 조기 차단해\n#            NLLB 에 "헤더+인사말" 이 혼합된 채 전달되는 것을 방지.\n_HEADER_ONLY = re.compile(\n    r"^[^.,!?~]{2,40}(안내|공지|알림|공개수업|상담|학습|행사|일정)\\s*$"\n)\n\n\ndef split_sentences(text: str) -> list:\n    lines = [line.strip() for line in text.splitlines() if line.strip()]\n    sentences = []\n    for line in lines:\n        if _HEADER_ONLY.match(line):   # 제목성 줄은 번역 대상에서 제외\n            continue\n        parts = re.split(\n            r"(?<=[.!?])\\s+|"\n            r"(?<=다\\.)\\s+|(?<=요\\.)\\s+|(?<=니다\\.)\\s+|"\n            r"(?<=까\\?)\\s+|(?<=요\\?)\\s+|"\n            r"\\s+(?=\\d+[.)]\\s)|\\s+(?=[가-힣]\\.\\s)",\n            line,\n        )\n        sentences.extend(parts)\n    return [s.strip() for s in sentences if s.strip() and len(s.strip()) > 3]\n\n\n# ── 3. 1차 필터 ──────────────────────────────────────────────────────────────\n# Bug 1 수정: "안녕하세요" 계열 패턴 추가\nNON_TODO_PATTERNS = [\n    r"^학부모님\\s*안녕하십니까",\n    r"^안녕하십니까",\n    r"^학부모님\\s*안녕하세요",        # Bug 1 추가\n    r"^안녕하세요",                    # Bug 1 추가\n    r"^.*님\\s*안녕하(세요|십니까)",   # Bug 1 추가 (일반화)\n    r"^학부모님께\\s*안내드립니다",\n    r"^학부모님께\\s*드립니다",\n    r"안내드립니다\\s*\\.?\\s*$",\n    r"드립니다\\s*\\.?\\s*$",\n    r"^[^.,!?]{1,30}\\s*안내\\s*$",\n    r"서울갈산초등학교장$",\n    r"교장$",\n    r"^\\d{4}\\.\\s*\\d{1,2}\\.\\s*\\d{1,2}\\.?\\s*$",\n    r"담당\\s*[:：]",\n    r"^\\(.\\s*\\d{4}-\\d{4}",\n    r"^08\\d{3}\\s*서울특별시",\n    r"공익제보센터",\n    r"자살예방상담",\n    r"청소년상담",\n]\n\n\ndef is_likely_todo(sentence: str) -> bool:\n    return not any(re.search(p, sentence) for p in NON_TODO_PATTERNS) and len(sentence) >= 7\n\n\n# ── 4. 정규식 기반 구조 추출 ─────────────────────────────────────────────────\nCURRENT_YEAR = 2026\n\nDATE_PATTERN_ABS = re.compile(\n    r"(?:(\\d{1,2})\\s*월\\s*(\\d{1,2})\\s*일)|"\n    r"(?<![\\d.])(?<!mm)(?<!cm)(?<!원)(?<!시)"\n    r"(\\d{1,2})[./](\\d{1,2})"\n    r"(?!\\d)(?![mc]m)(?!kg)"\n)\nDATE_PATTERN_REL = re.compile(\n    r"(다음\\s*주\\s*[월화수목금토일]요일|"\n    r"이번\\s*주\\s*[월화수목금토일]요일|"\n    r"매주\\s*[월화수목금토일]요일|"\n    r"오늘|내일|모레)"\n)\nDEADLINE_PATTERN = re.compile(r"([\\w가-힣\\s]+?)\\s*까지")\n\n# Bug 3 메모:\n#   MONEY_PATTERN 은 이미 \\d+\\s*원 형태 → "원하시는" 오탐 없음 (extraction 레벨 정상).\n#   검수 상세 "원→won" 오탐은 translation_tts/run_mvp_pipeline.py 글로사리 로직 문제.\n#   해당 파일에서 단순 str.contains("원") 대신\n#   re.search(r"\\d[\\d,]*\\s*원", text) 로 교체하면 해결.\nMONEY_PATTERN = re.compile(r"(\\d{1,3}(?:,\\d{3})+|\\d+)\\s*원")\n\n\ndef extract_due_date(sentence: str) -> Optional[str]:\n    m = DATE_PATTERN_ABS.search(sentence)\n    if m:\n        month = m.group(1) or m.group(3)\n        day   = m.group(2) or m.group(4)\n        if month and day:\n            try:\n                return f"{CURRENT_YEAR}-{int(month):02d}-{int(day):02d}"\n            except ValueError:\n                pass\n    m = DATE_PATTERN_REL.search(sentence)\n    if m:\n        return m.group(1).strip()\n    m = DEADLINE_PATTERN.search(sentence)\n    if m:\n        t = m.group(1).strip()\n        if t and len(t) < 20:\n            return t\n    return None\n\n\ndef has_money(sentence: str) -> bool:\n    return bool(MONEY_PATTERN.search(sentence))\n\n\n# ── 5. KoELECTRA 카테고리 분류 ───────────────────────────────────────────────\ndef classify_category(sentence: str) -> tuple:\n    _load_model()\n    inputs = _tokenizer(\n        sentence, return_tensors="pt",\n        truncation=True, padding=True, max_length=128,\n    ).to(_device)\n    with torch.no_grad():\n        logits = _model(**inputs).logits\n        probs  = torch.softmax(logits, dim=-1)[0]\n        pred_id = int(torch.argmax(probs).item())\n    return _id2label[pred_id], float(probs[pred_id].item())\n\n\n# ── 6. importance 계산 ────────────────────────────────────────────────────────\nCATEGORY_BASE_IMPORTANCE = {\n    "제출": 1.0, "준비물": 0.85, "건강·안전": 0.80,\n    "비용": 0.75, "일정": 0.70, "기타": 0.50,\n}\nURGENT_KEYWORDS = [\n    "반드시", "꼭", "필수", "엄수", "마감", "당일", "즉시",\n    "응급", "위급", "112", "신고",\n]\n\n\ndef calc_importance(sentence: str, category: str, due_date: Optional[str]) -> float:\n    score = CATEGORY_BASE_IMPORTANCE.get(category, 0.5)\n    if any(kw in sentence for kw in URGENT_KEYWORDS):\n        score = min(1.0, score + 0.05)\n    if due_date:\n        score = min(1.0, score + 0.05)\n    if len(sentence) > 80:\n        score = max(0.0, score - 0.05)\n    return round(score, 2)\n\n\n# ── 7. 메인 함수 ──────────────────────────────────────────────────────────────\ndef extract_todos(notice_text: str) -> list:\n    if not notice_text or not notice_text.strip():\n        return []\n    todos = []\n    for sent in [s for s in split_sentences(notice_text) if is_likely_todo(s)]:\n        due_date = extract_due_date(sent)\n        is_money = has_money(sent)\n        if is_money:\n            category, confidence = "비용", 1.0\n        else:\n            category, confidence = classify_category(sent)\n        if confidence < 0.25 and not is_money:\n            continue\n        importance = calc_importance(sent, category, due_date)\n        if importance < 0.25:\n            continue\n        text_ko = sent[:97] + "..." if len(sent) > 100 else sent\n        if SCHEMAS_AVAILABLE:\n            try:\n                todos.append(TodoItem(\n                    category=Category(category),\n                    text_ko=text_ko, text_vi="",\n                    importance=importance, due_date=due_date,\n                ))\n            except Exception as e:\n                print(f"[WARN] TodoItem 생성 실패: {e}", file=sys.stderr)\n                continue\n        else:\n            todos.append({\n                "category": category, "text_ko": text_ko,\n                "text_vi": "", "importance": importance, "due_date": due_date,\n            })\n    return todos\n\n\ndef extract_todos_dict(notice_text: str) -> list:\n    items = extract_todos(notice_text)\n    if SCHEMAS_AVAILABLE and items and isinstance(items[0], TodoItem):\n        return [item.model_dump() for item in items]\n    return items\n\n\nif __name__ == "__main__":\n    sample = """학부모 공개수업 및 상담 안내\n학부모님 안녕하세요.\n1. 공개수업 일시: 6월 12일(목) 3~4교시\n2. 상담 신청: 6월 5일(금)까지 가정통신문 회신\n3. 준비물: 실내화, 출입증 지참\n수업료 50,000원은 6월 10일까지 납부해 주세요.\n서울갈산초등학교장"""\n    print("=" * 60)\n    for i, t in enumerate(extract_todos_dict(sample), 1):\n        print(f"{i}. [{t[\'category\']}] importance={t[\'importance\']} due={t[\'due_date\']}")\n        print(f"   {t[\'text_ko\']}")\n    print("=" * 60)\n'

out = pathlib.Path("./predict.py")
out.write_text(PREDICT_PY, encoding="utf-8")
print(f"predict.py 생성 완료 → {out.resolve()}")
print(f"  크기: {out.stat().st_size:,} bytes")

## 12. 추론 테스트 — 버그 수정 확인

KoELECTRA 모델 없이도 돌아가는 **단위 테스트**. 수정된 패턴을 바로 검증.

1. **Bug 1**: `학부모님 안녕하세요.` → TODO 제외 확인
2. **Bug 2**: `학부모 공개수업 및 상담 안내` 헤더 줄 → split 단계에서 제거 확인
3. **Bug 3**: `원하시는`, `원인` → `has_money()` = False 확인 (MONEY_PATTERN 정상)

In [ ]:
import re

# ── Bug 1 ─────────────────────────────────────────────────────────────────────
NON_TODO_PATTERNS_FIXED = [
    r"^학부모님\s*안녕하십니까",
    r"^안녕하십니까",
    r"^학부모님\s*안녕하세요",
    r"^안녕하세요",
    r"^.*님\s*안녕하(세요|십니까)",
    r"^학부모님께\s*안내드립니다",
    r"^학부모님께\s*드립니다",
    r"안내드립니다\s*\.?\s*$",
    r"드립니다\s*\.?\s*$",
    r"^[^.,!?]{1,30}\s*안내\s*$",
    r"서울갈산초등학교장$",
    r"교장$",
    r"^\d{4}\.\s*\d{1,2}\.\s*\d{1,2}\.?\s*$",
    r"담당\s*[:：]",
]


def is_likely_todo_fixed(s):
    return not any(re.search(p, s) for p in NON_TODO_PATTERNS_FIXED) and len(s) >= 7


bug1_cases = [
    ("학부모님 안녕하세요.",                False, "Bug1: 인사말"),
    ("안녕하세요.",                          False, "Bug1: 짧은 인사말"),
    ("학부모님 안녕하십니까.",               False, "기존 패턴 유지"),
    ("6월 5일까지 가정통신문 회신해주세요.", True,  "정상 TODO"),
    ("준비물: 실내화, 출입증 지참",           True,  "준비물 TODO"),
    ("2026. 4. 17.",                         False, "날짜 서명"),
]

print("[ Bug 1 — NON_TODO_PATTERNS 수정 확인 ]")
all_ok = True
for sentence, expected, label in bug1_cases:
    got = is_likely_todo_fixed(sentence)
    ok = got == expected
    all_ok = all_ok and ok
    print(f"  {'✅' if ok else '❌'} {label}: is_todo={got} (expected {expected})")
print("  →", "전체 PASS ✅" if all_ok else "일부 FAIL ❌")

# ── Bug 2 ─────────────────────────────────────────────────────────────────────
_HEADER_ONLY = re.compile(
    r"^[^.,!?~]{2,40}(안내|공지|알림|공개수업|상담|학습|행사|일정)\s*$"
)

bug2_cases = [
    ("학부모 공개수업 및 상담 안내",         True,  "공개수업 헤더"),
    ("현장학습 안내",                         True,  "현장학습 헤더"),
    ("방과후 영어 수업 일정",                True,  "일정 헤더"),
    ("6월 12일 공개수업 참석 부탁드립니다.", False, "일반 문장 (통과)"),
]

print("\n[ Bug 2 — 헤더 줄 필터 확인 ]")
all_ok2 = True
for line, expected_filter, label in bug2_cases:
    got = bool(_HEADER_ONLY.match(line))
    ok = got == expected_filter
    all_ok2 = all_ok2 and ok
    print(f"  {'✅' if ok else '❌'} {label}: filtered={got} (expected {expected_filter})")
print("  →", "전체 PASS ✅" if all_ok2 else "일부 FAIL ❌")

# ── Bug 3 ─────────────────────────────────────────────────────────────────────
MONEY_PATTERN = re.compile(r"(\d{1,3}(?:,\d{3})+|\d+)\s*원")

bug3_cases = [
    ("원하시는 날짜에 오세요.",        False, "Bug3: '원하시는' 오탐 없음"),
    ("원인 파악이 필요합니다.",         False, "Bug3: '원인' 오탐 없음"),
    ("수업료 50,000원을 납부하세요.",  True,  "정상 금액 매칭"),
    ("500원 잔돈 준비",                True,  "소액 금액 매칭"),
]

print("\n[ Bug 3 — MONEY_PATTERN 확인 (extraction 레벨) ]")
all_ok3 = True
for text, expected, label in bug3_cases:
    got = bool(MONEY_PATTERN.search(text))
    ok = got == expected
    all_ok3 = all_ok3 and ok
    print(f"  {'✅' if ok else '❌'} {label}: has_money={got} (expected {expected})")
print("  →", "전체 PASS ✅" if all_ok3 else "일부 FAIL ❌")

print()
print("※ Bug 3 검수 상세 '원→won' 오탐 본체:")
print("  translation_tts/run_mvp_pipeline.py 에서")
print(r"  str.contains('원') → re.search(r'\d[\d,]*\s*원', text) 로 교체 필요.")


## 10. 압축 + 다운로드
다운받은 zip 을 풀어서  폴더에 두면  가 자동으로 로드해요.
 도 함께 zip 에 포함되어 있어요.

In [10]:
!zip -r koelectra-extractor.zip koelectra-extractor predict.py

from google.colab import files
files.download("koelectra-extractor.zip")

  adding: koelectra-extractor/ (stored 0%)
  adding: koelectra-extractor/training_args.bin (deflated 53%)
  adding: koelectra-extractor/special_tokens_map.json (deflated 42%)
  adding: koelectra-extractor/vocab.txt (deflated 49%)
  adding: koelectra-extractor/labels.json (deflated 55%)
  adding: koelectra-extractor/tokenizer.json (deflated 70%)
  adding: koelectra-extractor/config.json (deflated 55%)
  adding: koelectra-extractor/tokenizer_config.json (deflated 75%)
  adding: koelectra-extractor/pytorch_model.bin (deflated 8%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 끝

다운받은  을 로컬에 풀고,
 를  에 두면 백엔드가 자동 로드해요.

### 이번 수정 내역

| 항목 | 변경 전 | 변경 후 |
|------|---------|--------|
| 학습 에폭 | 10 | 15 |
| 학습률 | 3e-5 | 2e-5 |
| 스케줄러 | linear | cosine + warmup 10% |
| 손실 함수 | CrossEntropy (균등) | WeightedCrossEntropy (balanced) |
| NON_TODO_PATTERNS | 안녕하십니까만 | 안녕하세요 계열 3개 추가 |
| 헤더 필터 | 없음 | split_sentences() 에서 제목성 줄 조기 차단 |
| Bug 3 | - | translation_tts/run_mvp_pipeline.py 수정 필요 (주석 안내) |